In [2]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.schema import Document
from langchain_core.output_parsers import StrOutputParser

In [4]:
loader = TextLoader("langchain_sample.txt")
raw_docs = loader.load()

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a traditional 

In [6]:
query = "How can i use langchain to build an application with memory and tools?"

In [9]:
import huggingface_hub

In [11]:
import sys

!{sys.executable} -m pip install sentence-transformers faiss-cpu


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name= 'all-MiniLM-L6-v2')
vectorstore= FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k":8})

C:\Users\raavi\AppData\Local\Temp\ipykernel_20456\2985161477.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name= 'all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
## OpenAI Embedding
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
from langchain_openai import OpenAIEmbeddings

embeddings=OpenAIEmbeddings()
vectorstore_openai=FAISS.from_documents(docs,embeddings)
retriever_openai=vectorstore_openai.as_retriever(search_kwargs={"k":8})


In [17]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023BB7416450>, search_kwargs={'k': 8})

In [18]:
retriever_openai

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023C49862570>, search_kwargs={'k': 8})

In [21]:
import sys

!{sys.executable} -m pip uninstall -y langchain-groq
!{sys.executable} -m pip install langchain-groq==0.3.7

Found existing installation: langchain-groq 1.1.2
Uninstalling langchain-groq-1.1.2:
  Successfully uninstalled langchain-groq-1.1.2



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
### PROMPT
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
llm = init_chat_model(
    "gpt-5.4-mini",
    model_provider="openai"
)

llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000023C4C5A2ED0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023C4B3079E0>, root_client=<openai.OpenAI object at 0x0000023C4B2A6F60>, root_async_client=<openai.AsyncOpenAI object at 0x0000023C4B2A6E40>, model_name='gpt-5.4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True)

In [24]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")

In [25]:
retriever_docs =retriever.invoke(query)
retriever_docs

[Document(id='04c0c158-fcea-4eaa-b691-3767f73c2f41', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='22b2e7b1-c3a4-4dac-938f-94933a59077c', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='8b5e8404-59b5-4ae6-a113-5450adc21bfa', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging F

In [34]:
chain = prompt | llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000023C4C5A2ED0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023C4B3079E0>, root_client=<openai.OpenAI object at 0x0000023C4B2A6F60>, root_async_client=<openai.AsyncOpenAI object at 0x0000023C4B2A6E40>, model_name='gpt-5.4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_us

In [29]:
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retriever_docs)]
formatted_docs = "\n".join(doc_lines)

In [30]:
doc_lines

['1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '2. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '3. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and comp

In [31]:
formatted_docs

'1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.\n2. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\n3. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\n4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed ind

In [35]:
response=chain.invoke({"question":query,"documents":formatted_docs})
response

'2,1,4,5,6,3'

In [36]:
chain = prompt | llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000023C4C5A2ED0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023C4B3079E0>, root_client=<openai.OpenAI object at 0x0000023C4B2A6F60>, root_async_client=<openai.AsyncOpenAI object at 0x0000023C4B2A6E40>, model_name='gpt-5.4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_us